# 00 — Data Profiling
**Coffee Consumption Around the World**

Before building any pipeline, we inspect the three raw sources with our own eyes. The goal is not to transform anything yet — it is to answer one question:

> *What is inside each file, and where will the three sources disagree when we try to join them?*

Everything downstream (the country mapping, the schema, the marts) depends on what we learn here. This notebook is also **evidence of process**: it shows we investigated before we built.

**The three sources:**
| File | Source | Role |
|---|---|---|
| `psd_coffee.csv` | USDA PSD | coffee production/consumption/trade, by country & year |
| `population.csv` | World Bank | total population, by country & year |
| `countries-codes.csv` | OpenDataSoft | ISO country codes — our canonical "spine" |

## Cell 1 — Load all three files

**What we're doing:** reading each CSV into a pandas DataFrame (a table in memory) and printing its size, so we know the load worked and how big each source is.

**Line by line:**
- `import pandas as pd, unicodedata, re` — pandas for tables; `unicodedata` and `re` (regex) are for cleaning country-name text later.
- `pd.set_option("display.max_columns", None)` — tells pandas to show *all* columns when we preview, not truncate them.
- `pd.read_csv("../data/raw/psd_coffee.csv")` — loads the coffee file. The `../` means "go up one folder": the notebook lives in `notebooks/`, the data lives in `data/raw/`, so we step up then back down.
- `pd.read_csv("../data/raw/population.csv", skiprows=4)` — the World Bank file has **4 junk metadata rows** above the real header (title, last-updated date, blank lines). `skiprows=4` skips them so the true header row is used.
- `pd.read_csv("../data/raw/countries-codes.csv", sep=";")` — this file is **semicolon-delimited**, not comma. More on why below.
- The `.rename(...)` — standardizes the codes file's column names to clean lowercase names we control (explained in the error notes below).

In [37]:
import pandas as pd, unicodedata, re
pd.set_option("display.max_columns", None)

coffee = pd.read_csv("../data/raw/psd_coffee.csv")
pop    = pd.read_csv("../data/raw/population.csv", skiprows=4)
codes  = pd.read_csv("../data/raw/countries-codes.csv", sep=";")

# Standardize the codes columns to names we control (this export uses UPPERCASE with spaces)
codes = codes.rename(columns={
    "ISO2 CODE": "iso2_code",
    "ISO3 CODE": "iso3_code",
    "LABEL EN":  "label_en",
})

print("coffee:", coffee.shape)
print("pop:   ", pop.shape)
print("codes: ", codes.shape)

coffee: (85937, 12)
pop:    (265, 71)
codes:  (247, 11)


### ⚠️ Two errors we hit on this cell — and how we fixed them

**Error 1 — `ParserError: Expected 3132 fields... saw 3288`**
When we tried reading the codes file with a comma separator, pandas exploded. Cause: the file contains a giant `geo_shape` column full of map coordinates that themselves contain **thousands of commas**. Comma-splitting therefore counted wildly different numbers of fields per row.
**Fix:** the file's real delimiter is a semicolon (`;`) — chosen precisely so the commas inside coordinates don't break parsing. Adding `sep=";"` resolved it.

**Error 2 — `KeyError: 'iso2_code'`**
After fixing the delimiter, later cells failed because the columns weren't named `iso2_code`/`iso3_code`/`label_en`. This OpenDataSoft export names them in **UPPERCASE with spaces**: `ISO2 CODE`, `ISO3 CODE`, `LABEL EN`.
**Fix:** rename them to clean lowercase names immediately after loading, so every downstream cell uses consistent, predictable names.

**Lesson:** public data sources have unstable delimiters and column names. Catching this here (in throwaway cells) instead of deep in the pipeline is exactly why profiling exists — and it's why our real `extract.py` will pin the delimiter and rename columns explicitly, failing loudly if an expected column is missing.

## Cell 2 — Inspect the USDA coffee data (shape, columns, grain)

**What we're doing:** understanding the structure of the most complex source. Specifically we want the columns, how many countries it covers, the year range, the units, and the full list of "attributes."

**Line by line:**
- `coffee.columns.tolist()` — lists the column names.
- `coffee["Country_Name"].nunique()` — counts distinct countries.
- `coffee["Market_Year"].min()/.max()` — the earliest and latest year.
- `coffee["Unit_Description"].unique()` — the measurement unit (critical: we must convert it later).
- the loop over `Attribute_Description` — prints every attribute (Production, Domestic Consumption, Imports, …).
- `coffee.head(3)` — shows the first 3 real rows.

In [29]:
print("columns:", coffee.columns.tolist())
print("\ndistinct countries:", coffee["Country_Name"].nunique())
print("year range:", coffee["Market_Year"].min(), "-", coffee["Market_Year"].max())
print("units:", coffee["Unit_Description"].unique())
print("\nattributes:")
for a in sorted(coffee["Attribute_Description"].unique()):
    print("  ", a)
coffee.head(3)

columns: ['Commodity_Code', 'Commodity_Description', 'Country_Code', 'Country_Name', 'Market_Year', 'Calendar_Year', 'Month', 'Attribute_ID', 'Attribute_Description', 'Unit_ID', 'Unit_Description', 'Value']

distinct countries: 94
year range: 1960 - 2025
units: <ArrowStringArray>
['(1000 60 KG BAGS)']
Length: 1, dtype: str

attributes:
   Arabica Production
   Bean Exports
   Bean Imports
   Beginning Stocks
   Domestic Consumption
   Ending Stocks
   Exports
   Imports
   Other Production
   Production
   Roast & Ground Exports
   Roast & Ground Imports
   Robusta Production
   Rst,Ground Dom. Consum
   Soluble Dom. Cons.
   Soluble Exports
   Soluble Imports
   Total Distribution
   Total Supply


,Commodity_Code,Commodity_Description,Country_Code,Country_Name,Market_Year,Calendar_Year,Month,Attribute_ID,Attribute_Description,Unit_ID,Unit_Description,Value
0,711100,"Coffee, Green",AL,Albania,2005,2023,6,29,Arabica Production,2,(1000 60 KG BAGS),0.0
1,711100,"Coffee, Green",AL,Albania,2005,2023,6,90,Bean Exports,2,(1000 60 KG BAGS),0.0
2,711100,"Coffee, Green",AL,Albania,2005,2023,6,58,Bean Imports,2,(1000 60 KG BAGS),80.0


**Finding:** ~94 countries, decades of years, units in **"1000 60 KG BAGS"** (so every value must be multiplied to get kilograms later). Crucially, the **grain** is one row = *country + year + attribute*, NOT one row per country. So "Brazil 2020" is spread across ~19 rows (one per attribute). This "long" shape drives a reshaping decision in the pipeline.

## Cell 3 — THE TRAP: is USDA's `Country_Code` the same as ISO2?

**What we're doing:** USDA has a `Country_Code` column that *looks* joinable to our ISO2 spine. Before trusting it, we **test** it — join USDA's code to the spine's `iso2_code` and see how often the resulting country name actually matches.

**Line by line:**
- `coffee[["Country_Code","Country_Name"]].drop_duplicates()` — one row per USDA country (its code + name).
- `.merge(codes, left_on="Country_Code", right_on="iso2_code", how="left")` — attempt the join on the code. `how="left"` keeps every USDA row even when no match is found (so we can count the failures).
- the `wrong = ...` filter — flags rows where either **no match was found** (`label_en` is null) OR the **matched name disagrees** with USDA's name.
- the prints — show how many matched correctly, and examples of the wrong matches.

In [30]:
usda = coffee[["Country_Code","Country_Name"]].drop_duplicates()
test = usda.merge(codes, left_on="Country_Code", right_on="iso2_code", how="left")
wrong = test[test["label_en"].isna() |
             (test["Country_Name"].str.lower() != test["label_en"].str.lower())]
print(f"USDA code == ISO2 correct for only {len(usda)-len(wrong)}/{len(usda)} countries")
print("\nExamples where the code-join gives the WRONG country:")
wrong[["Country_Code","Country_Name","label_en"]].head(15)

USDA code == ISO2 correct for only 35/94 countries

Examples where the code-join gives the WRONG country:


,Country_Code,Country_Name,label_en
1,AG,Algeria,Antigua and Barbuda
5,AS,Australia,American Samoa
6,DM,Benin,Dominica
7,BL,Bolivia,NaN
8,BK,Bosnia and Herzegovina,NaN
10,BY,Burundi,Belarus
13,CT,Central African Republic,NaN
14,CI,Chile,Côte d'Ivoire
15,CH,China,Switzerland
17,CF,Congo (Brazzaville),Central African Republic


**Finding:** the code matches ISO2 for only about **35 of 94** countries. The mismatches are alarming — e.g. China→Switzerland, Nigeria→Nicaragua, Algeria→Antigua & Barbuda. USDA's `Country_Code` is **FIPS 10-4**, a different standard from ISO. **Decision: reject the code column entirely; join USDA to the spine by country NAME instead.** This is our single most important design decision, and now we have hard evidence for it rather than an assumption.

## Cell 4 — Match USDA names to the spine (normalize + exact match)

**What we're doing:** since we're joining by name, we test how many USDA names match the spine's English labels after light cleaning — and list the ones that DON'T (those need manual mapping).

**Line by line:**
- `normalize(s)` — a cleaning function: strips accents (so `Côte d'Ivoire` ≡ `Cote d Ivoire`), lowercases, removes punctuation, collapses spaces. This makes matching robust to trivial formatting differences.
- `spine = dict(zip(codes["norm"], codes["iso3_code"]))` — builds a lookup: normalized-name → ISO3 code.
- `hits` / `misses` — split USDA names into those that match the spine and those that don't.
- the prints — the match count and the exact list of unmatched names.

In [31]:
def normalize(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s.lower())).strip()

codes["norm"] = codes["label_en"].map(normalize)
spine = dict(zip(codes["norm"], codes["iso3_code"]))

names = sorted(coffee["Country_Name"].unique())
hits   = [n for n in names if normalize(n) in spine]
misses = [n for n in names if normalize(n) not in spine]

print(f"exact-after-normalize: {len(hits)}/{len(names)} matched")
print(f"\n{len(misses)} names need manual overrides:")
for n in misses:
    print("  ", n)

exact-after-normalize: 81/94 matched

13 names need manual overrides:
   Congo (Brazzaville)
   Congo (Kinshasa)
   European Union
   Iran
   Korea, South
   Laos
   North Macedonia
   Russia
   Taiwan
   Tanzania
   Venezuela
   Vietnam
   Yemen (Sanaa)


**Finding:** roughly **81 of 94** names match automatically. The remaining ~13 are naming variants the spine writes differently (e.g. Russia→"Russian Federation", Vietnam→"Viet Nam", Korea, South→"Korea, Republic of") plus **European Union**, which is an aggregate, not a country. This ~13-name list becomes our hand-curated `mappings/country_overrides.csv` in the next phase — with European Union flagged for **exclusion**.

## Cell 5 — World Bank: find the aggregate rows that aren't real countries

**What we're doing:** the population file mixes real countries with regional/economic **aggregates** ("World", "European Union", "High income"). These carry real-looking ISO3 codes and would poison per-capita math if treated as countries. We find them by comparing World Bank's codes against our spine's real-country codes.

**Line by line:**
- `wb_codes` — the set of all codes in the World Bank file.
- `spine_iso3` — the set of real-country codes from the spine.
- `wb_codes - spine_iso3` — set subtraction: codes in World Bank that are NOT real countries.
- the filter + print — list those aggregate rows.

In [32]:
wb_codes    = set(pop["Country Code"])
spine_iso3  = set(codes["iso3_code"].dropna())
aggregates  = pop[pop["Country Code"].isin(wb_codes - spine_iso3)][["Country Name","Country Code"]]

print(f"World Bank total rows: {len(pop)}")
print(f"Rows that are NOT real countries (aggregates): {len(aggregates)}")
aggregates.head(30)

World Bank total rows: 265
Rows that are NOT real countries (aggregates): 54


,Country Name,Country Code
1,Africa Eastern and Southern,AFE
3,Africa Western and Central,AFW
7,Arab World,ARB
36,Central Europe and the Baltics,CEB
38,Channel Islands,CHI
49,Caribbean small states,CSS
51,Curacao,CUW
61,East Asia & Pacific (excluding high income),EAP
62,Early-demographic dividend,EAR
63,East Asia & Pacific,EAS


**Finding:** about **54** of 265 rows are aggregates — "World" (WLD), "European Union" (EUU), income groups (HIC, LIC…), regions. **Decision:** exclude these by using the spine as an **allowlist** — nothing enters a fact table unless its ISO3 exists in our country dimension. In the database this is enforced structurally by a foreign key, so an aggregate can't sneak in even if a filter is forgotten.

## Cell 6 — World Bank is in WIDE format (needs melting)

**What we're doing:** confirming the population file has one **column per year** (wide), which must be reshaped into tidy rows of (country, year, population) before loading.

**Line by line:**
- printing the first several column names — reveals `1960, 1961, 1962…` as columns.
- `pop.iloc[:3, :8]` — shows the first 3 rows and first 8 columns so the wide layout is visible.

In [33]:
print("first columns:", pop.columns.tolist()[:8], "...")
print("\nOne column per year = WIDE format. Must melt to (iso3, year, population) before loading.")
pop.iloc[:3, :8]

first columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963'] ...

One column per year = WIDE format. Must melt to (iso3, year, population) before loading.


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0


In [34]:
coffee[coffee["Country_Name"].str.contains("Yemen")].groupby("Country_Name")["Market_Year"].agg(["min","max"])

,min,max
Country_Name,,
Yemen,1991,2025
Yemen (Sanaa),1960,1990


## Data quality check: Yemen double-counting

USDA carries two Yemen entities. Verified their year ranges don't overlap:
- **Yemen (Sanaa)**: 1960–1990 (historical North Yemen)
- **Yemen**: 1991–2025 (modern unified Yemen)

They hand off at the 1990 unification, so mapping both to ISO3 **YEM** 
stitches one continuous national timeline — no double-counting.

**Finding:** confirmed wide. The pipeline will `melt` this into long, tidy rows so each (country, year) is one record — the shape a database wants.

## Summary — what profiling established

| Source | Key facts | Decisions |
|---|---|---|
| **USDA coffee** | 94 countries; long format; units = 1000×60kg bags; `Country_Code` is a FIPS **decoy** | join by **name**; convert units once; reshape long→wide |
| **name matching** | ~81/94 auto-match | ~13 manual overrides + exclude "European Union"; add a **gate** for future surprises |
| **World Bank pop** | wide format; 4 junk header rows; ~54 aggregate rows | `skiprows=4`; **melt**; exclude aggregates via spine **allowlist** |
| **OpenDataSoft codes** | semicolon-delimited; UPPERCASE column names | the canonical **ISO3 spine / dimension** |

**Errors met & resolved:** (1) `ParserError` → wrong delimiter, fixed with `sep=";"`; (2) `KeyError` → non-standard column names, fixed with an explicit rename. Both foreshadow why the real pipeline validates delimiters and columns explicitly.

**The join strategy in one line:**
> All three sources join through **ISO3** as the spine. World Bank joins cleanly by code (after excluding 54 aggregates); USDA must join by **name** (its code is a FIPS decoy), needing ~13 manual overrides plus a gate to catch future mismatches.

**Next phase:** build `pipeline/resolve.py` and `mappings/country_overrides.csv` — the country mapping, the most heavily-graded part of the assignment.